# 01 · Pandas API on Spark

**Goal:** explore and prepare a large table with the pandas commands you already know, while Spark does the work
across many machines.

**Why it matters:** plain pandas loads everything into one computer's memory. When the data is bigger than that, pandas
fails. `pyspark.pandas` keeps the familiar syntax (`head`, `describe`, `groupby`, `fillna`) but runs it on Spark.

| | pandas | Pandas API on Spark |
|---|---|---|
| Where the data lives | One machine's memory | Spread across the cluster |
| When work happens | Immediately | Only when a result is needed (**lazy**) |
| Import | `import pandas as pd` | `import pyspark.pandas as ps` |

In [ ]:
%run ./Includes/_config

## 1. From a Spark table to a pandas-on-Spark DataFrame
`.pandas_api()` turns a Spark DataFrame into a pandas-on-Spark DataFrame. Nothing is copied to your notebook.

In [ ]:
import pyspark.pandas as ps

psdf = spark.table(TABLE).pandas_api()
print(type(psdf))
print("Rows, columns:", psdf.shape)
psdf.head()

## 2. Inspect with familiar commands

In [ ]:
psdf[["BMI", "Age", "GenHlth", "MentHlth", "PhysHlth"]].describe()

In [ ]:
# Share of people with diabetes in each age group (Age is a 1–13 code: 1 = 18–24 ... 13 = 80+)
psdf.groupby("Age")[LABEL].mean().sort_index()

## 3. Feature engineering with pandas syntax
New columns are defined exactly as in pandas. Spark records the steps and runs them later, in parallel.

In [ ]:
psdf["obese"] = (psdf["BMI"] >= 30).astype("int")
psdf["poor_health_days"] = psdf["MentHlth"] + psdf["PhysHlth"]
psdf["risk_factors"] = psdf["HighBP"] + psdf["HighChol"] + psdf["obese"] + psdf["Smoker"]

# Pandas API on Spark covers most, but not all, of pandas. A list of functions on one column
# (.agg(["mean", "count"])) is not supported yet, so aggregate with a dictionary instead:
(psdf.groupby("risk_factors")
     .agg({LABEL: "mean", "id": "count"})
     .rename(columns={LABEL: "diabetes_rate", "id": "people"})
     .sort_index())

## 4. Lazy evaluation: see the plan Spark built
Converting to a Spark DataFrame shows the optimised plan behind your pandas code.

In [ ]:
features_sdf = psdf.to_spark()
features_sdf.select("risk_factors", "poor_health_days", "obese").explain()

## 5. Bringing data back to one machine safely
`to_pandas()` copies **everything** to your notebook. On large data, take a **sample** first.

In [ ]:
sample_pdf = psdf.sample(frac=0.02, random_state=42).to_pandas()
print(type(sample_pdf), sample_pdf.shape)

# Save the engineered features as a table for the next notebooks
features_sdf.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.diabetes_features")
print("Saved", f"{CATALOG}.{SCHEMA}.diabetes_features")

## Key points
- `pyspark.pandas` = pandas syntax, Spark scale.
- Work is **lazy**: Spark plans first and runs when a result is needed.
- Sample before `to_pandas()`; never pull a huge table onto one machine.
- Switch freely: `.pandas_api()` (Spark → pandas-on-Spark) and `.to_spark()` (back to Spark).
- A few pandas features are not implemented yet (for example a list of functions in `.agg` on one column). The error
  message says so clearly, and there is usually an equivalent form, such as a dictionary in `.agg`.